# Projeto de Engenharia 01 — Classificação de Pureza de Óleo com `Perceptron`

## Contexto de Engenharia Química e de Petróleo

Em uma coluna de **destilação fracionada de petróleo**, deseja-se classificar automaticamente lotes de óleo lubrificante em duas classes de pureza comercial (`P1` e `P2`) medindo três propriedades físico-químicas em linha ($x_1$: viscosidade relativa, $x_2$: índice de acidez, $x_3$: ponto de fulgor normalizado).

A partir de ensaios históricos montou-se a tabela de referência `../arquivos-csv/tabela_apendice_i.csv` adotando a convenção bipolar:
- **Classe P1:** `d = -1`
- **Classe P2:** `d = +1`

### Objetivos do Projeto
1. Treinar **5 redes Perceptron** (`T1` a `T5`) iniciando os pesos $\mathbf{w} = [w_0, w_1, w_2, w_3]$ com sorteios distintos entre `0` e `1`.
2. Registrar pesos iniciais, pesos finais e número de épocas necessárias para convergir a **erro zero**.
3. Visualizar em **3D** como os 5 planos aleatórios iniciais se organizam após o treinamento em um único corredor de separação!
4. Classificar **10 amostras inéditas de planta** (`../arquivos-csv/amostras_oleo.csv`) e verificar se os 5 modelos concordam entre si.

---

## Passo 1: Carregamento e Inspeção das Amostras de Treinamento

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Gerador aleatório moderno com semente fixa (garante reprodutibilidade dos 5 sorteios)
rng = np.random.default_rng(42)

# Lê as 30 amostras de referência do Apêndice I
train_data = pd.read_csv("../arquivos-csv/tabela_apendice_i.csv")

print(f"Total de ensaios de treinamento: {len(train_data)}")
print("Contagem por classe (-1 = P1, +1 = P2):")
print(train_data["d"].value_counts())
print("\nPrimeiras 5 amostras:")
print(train_data.head())

**Análise Inicial:** O conjunto de treinamento possui `30` amostras equilibradas entre as classes `-1.0` (`P1`) e `+1.0` (`P2`), sem valores ausentes.

---

## Passo 2: Construção da Classe `Perceptron` Comentada Linha a Linha

Abaixo estruturamos o neurônio como um objeto `Python` reutilizável:
- O vetor `self.weights` guarda `[w0, w1, w2, w3]`, onde `w0` multiplica a entrada fixa `x0 = -1.0` (representando o limiar $\theta$).
- A função `degrau_bipolar(u)` implementa o relé de decisão (`+1` se $u \ge 0$, `-1` se $u < 0$).

In [ ]:
def degrau_bipolar(u):
    # Função de ativação degrau bipolar: retorna +1 para potencial não negativo e -1 caso contrário
    return 1 if u >= 0.0 else -1


class Perceptron:
    def __init__(self, pesos_iniciais, func_ativacao):
        # Fazemos .copy() para não alterar acidentalmente o vetor original guardado para relatório
        self.weights = pesos_iniciais.copy()
        self.activation_func = func_ativacao
        self.epochs = 0              # Guardará quantas épocas levou até zerar os erros
        self.errors_history = []     # Guardará o número de erros cometidos em cada época

    def feedforward(self, entradas):
        # Potencial de ativação u = w0*(-1) + w1*x1 + w2*x2 + w3*x3
        return float(np.dot(self.weights, entradas))

    def predict(self, entradas):
        # Calcula o potencial u e aplica o degrau bipolar (+1 ou -1)
        u = self.feedforward(entradas)
        return self.activation_func(u)

    def train(self, dataframe, learning_rate=0.01, max_epochs=1000):
        # Percorre até max_epochs ciclos completos sobre as 30 amostras
        for epoca in range(1, max_epochs + 1):
            erros_na_epoca = 0

            # Avalia cada linha (ensaio) da tabela
            for _, linha in dataframe.iterrows():
                # Monta o vetor de entrada com x0 = -1 na frente das 3 medidas físico-químicas
                x = np.array([-1.0, linha["x1"], linha["x2"], linha["x3"]])
                d = int(linha["d"])

                # Previsão atual do neurônio
                y = self.predict(x)

                # Regra de correção de erro: só mexe nas válvulas (pesos) se errar!
                if y != d:
                    self.weights = self.weights + learning_rate * (d - y) * x
                    erros_na_epoca += 1

            self.errors_history.append(erros_na_epoca)

            # Se passou pelas 30 amostras sem errar nenhuma, atingiu convergência exata!
            if erros_na_epoca == 0:
                self.epochs = epoca
                break
        else:
            # Caso atinja max_epochs sem zerar erros
            self.epochs = max_epochs

        return self.epochs

---

## Passo 3: Executando os 5 Treinamentos Independentes (`T1` a `T5`)

Em cada execução sorteamos um novo vetor `pesos_ini = rng.random(4)` no intervalo `[0, 1)` e treinamos com taxa de aprendizado $\eta = 0{,}01$.

In [ ]:
resultados_treino = []
neuronios_treinados = []

for i in range(5):
    # Sorteia 4 valores uniformes entre 0 e 1 para [w0, w1, w2, w3]
    pesos_ini = rng.random(4)

    # Cria e treina o neurônio Perceptron
    neuronio = Perceptron(pesos_ini, degrau_bipolar)
    total_epocas = neuronio.train(train_data, learning_rate=0.01, max_epochs=2000)
    neuronios_treinados.append(neuronio)

    # Registra pesos iniciais, finais e épocas em dicionário para montar a tabela
    resultados_treino.append({
        "Treinamento": f"T{i + 1}",
        "w0_ini": pesos_ini[0], "w1_ini": pesos_ini[1], "w2_ini": pesos_ini[2], "w3_ini": pesos_ini[3],
        "w0_fim": neuronio.weights[0], "w1_fim": neuronio.weights[1],
        "w2_fim": neuronio.weights[2], "w3_fim": neuronio.weights[3],
        "Epocas": total_epocas,
    })

df_resultados = pd.DataFrame(resultados_treino)
print("Tabela de Convergência dos 5 Treinamentos:")
print(df_resultados.round(4).to_string(index=False))

**Interpretação dos Números:**
- Todos os **5 treinamentos convergiram para erro zero** entre ~395 e ~439 épocas, comprovando matematicamente que as duas classes de óleo são **linearmente separáveis** no espaço tridimensional $(x_1, x_2, x_3)$!
- Repare como os pesos finais ficaram muito próximos entre si (`w0 ≈ -3.1`, `w1 ≈ 1.56`, `w2 ≈ 2.49`, `w3 ≈ -0.74`), independentemente do ponto de partida aleatório.

Vamos plotar a queda do número de erros por época e comparar visualmente os **planos iniciais caóticos** com os **planos finais convergidos** no espaço 3D:

In [ ]:
# 1. Curva de redução de erros por época
plt.figure(figsize=(9, 4.5))
for i, neuronio in enumerate(neuronios_treinados):
    plt.plot(neuronio.errors_history, label=f"Treinamento T{i + 1}")
plt.title("Queda do Número de Erros de Classificação por Época")
plt.xlabel("Época")
plt.ylabel("Amostras Classificadas Erradas")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# 2. Comparação 3D lado a lado: Planos Iniciais (Aleatórios) vs. Planos Finais (Treinados)
fig = plt.figure(figsize=(14, 6))
ax_ini = fig.add_subplot(121, projection="3d")
ax_fim = fig.add_subplot(122, projection="3d")

classe_p1 = train_data[train_data["d"] == -1]
classe_p2 = train_data[train_data["d"] == 1]

# Malha (x1, x2) para desenhar a superfície do plano u = -w0 + w1*x1 + w2*x2 + w3*x3 = 0
x1_malha = np.linspace(train_data["x1"].min(), train_data["x1"].max(), 12)
x2_malha = np.linspace(train_data["x2"].min(), train_data["x2"].max(), 12)
X1, X2 = np.meshgrid(x1_malha, x2_malha)
cores = ["green", "purple", "orange", "teal", "brown"]

for ax, prefixo, titulo in [
    (ax_ini, "ini", "Antes do Treino (Planos Aleatórios)"),
    (ax_fim, "fim", "Após o Treino (Fronteiras Convergidas)")
]:
    ax.scatter(classe_p1["x1"], classe_p1["x2"], classe_p1["x3"], c="royalblue", marker="o", s=40, label="P1 (-1)")
    ax.scatter(classe_p2["x1"], classe_p2["x2"], classe_p2["x3"], c="crimson", marker="^", s=45, label="P2 (+1)")

    for i, linha in df_resultados.iterrows():
        w0 = linha[f"w0_{prefixo}"]
        w1 = linha[f"w1_{prefixo}"]
        w2 = linha[f"w2_{prefixo}"]
        w3 = linha[f"w3_{prefixo}"]
        # Isolando x3 na equação do plano -w0 + w1*x1 + w2*x2 + w3*x3 = 0:
        X3 = (w0 - w1 * X1 - w2 * X2) / w3
        ax.plot_surface(X1, X2, X3, color=cores[i], alpha=0.22)
        ax.plot([], [], [], color=cores[i], label=f"Plano T{i + 1}")

    ax.set_title(titulo)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_zlabel("$x_3$")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Interpretação Geométrica:** À esquerda, os 5 planos iniciais cortam as amostras de forma caótica; à direita, após o aprendizado, os 5 planos se alinham quase como uma única folha separando perfeitamente os pontos azuis (`P1`) dos triângulos vermelhos (`P2`)!

---

## Passo 4: Classificando as 10 Novas Amostras de Óleo da Planta Industrial

In [ ]:
# Lê as 10 amostras desconhecidas coletadas na coluna de destilação
validacao_df = pd.read_csv("../arquivos-csv/amostras_oleo.csv")

# Preenche as colunas y_T1 até y_T5 com a decisão de cada um dos 5 neurônios treinados
for i, neuronio in enumerate(neuronios_treinados):
    coluna_saida = f"y_T{i + 1}"
    previsoes = []
    for _, amostra in validacao_df.iterrows():
        x_novo = np.array([-1.0, amostra["x1"], amostra["x2"], amostra["x3"]])
        previsoes.append(neuronio.predict(x_novo))
    validacao_df[coluna_saida] = previsoes

print("Classificação das 10 Amostras Reais (-1 = Pureza P1 | +1 = Pureza P2):")
print(validacao_df.to_string(index=False))

**Conclusão de Engenharia:** Os **5 modelos concordaram em 100% das 10 amostras**! Isso acontece porque o corredor de separação entre `P1` e `P2` é estreito o suficiente para guiar todas as 5 inicializações para hiperplanos praticamente idênticos.

---

## Exercícios Práticos de Fixação

### Exercício 1: Influência da Taxa de Aprendizado ($\eta$) no Número de Épocas
Treine um novo `Perceptron` sobre `train_data` partindo dos mesmos pesos iniciais de `T1`, mas testando três taxas de aprendizado: `0.005`, `0.01` e `0.05`. Compare quantas épocas cada uma leva para convergir.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
pesos_base = df_resultados.loc[0, ["w0_ini", "w1_ini", "w2_ini", "w3_ini"]].to_numpy(dtype=float)

for taxa in [0.005, 0.01, 0.05]:
    p = Perceptron(pesos_base, degrau_bipolar)
    ep = p.train(train_data, learning_rate=taxa, max_epochs=5000)
    print(f"Taxa lr = {taxa:5.3f}  -->  Convergiu em {ep:4d} épocas | Pesos finais: {np.round(p.weights, 3)}")
```

</details>

### Exercício 2: Distância até a Fronteira de Decisão (Margem de Segurança)
Em controle de qualidade, amostras com potencial $|u|$ muito próximo de `0.0` estão "em cima do muro" (próximas ao plano divisor) e merecem reanálise laboratorial. Calcule o potencial escalar $u = \mathbf{w}_{T1} \cdot \mathbf{x}$ das 10 amostras de `validacao_df` e descubra qual amostra ficou **mais próxima da fronteira** ($|u|$ mínimo).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
neuronio_t1 = neuronios_treinados[0]

potenciais_abs = []
for _, amostra in validacao_df.iterrows():
    x_v = np.array([-1.0, amostra["x1"], amostra["x2"], amostra["x3"]])
    u = neuronio_t1.feedforward(x_v)
    potenciais_abs.append((int(amostra["Amostra"]), u, abs(u)))

# Ordena pelo módulo |u| crescente para achar a amostra mais próxima de u = 0
mais_proxima = min(potenciais_abs, key=lambda item: item[2])
print(f"Amostra mais próxima da fronteira: Amostra #{mais_proxima[0]} (u = {mais_proxima[1]:.4f})")
```

</details>